# Direct Postgres Connection — Hive Metastore
This notebook connects **directly** to the `hive-postgres` container using `psycopg2`.

No Spark needed — pure Python SQL queries against the PostgreSQL Hive Metastore.

| Setting | Value |
|---|---|
| Container | `hive-postgres` |
| Database | `metastore` |
| User | `hive` |
| Password | `hive` |

In [ ]:
# Step 1: Install psycopg2 if not already present
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', 'psycopg2-binary', '--quiet'], check=True)
print('psycopg2-binary ready.')

## Discover the Postgres Container IP

Since port `5432` is not published to the host in `docker-compose.yml`, we resolve the container's internal Docker bridge IP dynamically.

In [ ]:
import socket

# The notebook kernel runs inside the 'namenode' Docker container,
# which shares the same 'hadoop-network' as 'hive-postgres'.
# The hostname resolves directly — no docker CLI needed.
POSTGRES_HOST = "hive-postgres"

try:
    resolved_ip = socket.gethostbyname(POSTGRES_HOST)
    print(f"Resolved '{POSTGRES_HOST}' -> {resolved_ip}")
except socket.gaierror as e:
    raise RuntimeError(
        f"Cannot resolve hostname '{POSTGRES_HOST}': {e}\n"
        "Make sure the hive-postgres container is running."
    )

print(f"Using host: {POSTGRES_HOST}:5432")

## Connect to Postgres

In [ ]:
import psycopg2
import psycopg2.extras

conn = psycopg2.connect(
    host=POSTGRES_HOST,
    port=5432,
    dbname='metastore',
    user='hive',
    password='hive'
)
conn.autocommit = True
cursor = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)

print(f'Connected to PostgreSQL at {POSTGRES_HOST}:5432/metastore')

## List All Hive Databases

In [ ]:
import pandas as pd

cursor.execute('SELECT "DB_ID", "NAME", "DESC", "OWNER_NAME" FROM "DBS" ORDER BY "DB_ID";')
rows = cursor.fetchall()

df_dbs = pd.DataFrame(rows)
print(f'Found {len(df_dbs)} database(s)')
df_dbs

## List All Hive Tables

In [ ]:
cursor.execute('''
    SELECT
        d."NAME"      AS database_name,
        t."TBL_NAME"  AS table_name,
        t."OWNER"     AS owner,
        t."TBL_TYPE"  AS table_type
    FROM "TBLS" t
    JOIN "DBS"  d ON t."DB_ID" = d."DB_ID"
    ORDER BY d."NAME", t."TBL_NAME";
''')
rows = cursor.fetchall()

df_tables = pd.DataFrame(rows)
print(f'Found {len(df_tables)} table(s)')
df_tables

## List Columns for Each Table

In [ ]:
cursor.execute('''
    SELECT
        d."NAME"       AS database_name,
        t."TBL_NAME"   AS table_name,
        c."COLUMN_NAME",
        c."TYPE_NAME",
        c."INTEGER_IDX" AS col_order
    FROM "COLUMNS_V2" c
    JOIN "SDS"         s ON c."CD_ID"  = s."CD_ID"
    JOIN "TBLS"        t ON s."SD_ID"  = t."SD_ID"
    JOIN "DBS"         d ON t."DB_ID"  = d."DB_ID"
    ORDER BY d."NAME", t."TBL_NAME", c."INTEGER_IDX";
''')
rows = cursor.fetchall()

df_cols = pd.DataFrame(rows)
print(f'Found {len(df_cols)} column(s) across all tables')
df_cols

## Table Storage Locations (HDFS Paths)

In [ ]:
cursor.execute('''
    SELECT
        d."NAME"      AS database_name,
        t."TBL_NAME"  AS table_name,
        s."LOCATION"  AS hdfs_location,
        s."INPUT_FORMAT",
        s."OUTPUT_FORMAT"
    FROM "SDS"  s
    JOIN "TBLS" t ON s."SD_ID" = t."SD_ID"
    JOIN "DBS"  d ON t."DB_ID" = d."DB_ID"
    ORDER BY d."NAME", t."TBL_NAME";
''')
rows = cursor.fetchall()

df_loc = pd.DataFrame(rows)
df_loc

## Run Any Custom SQL

You can query any table in the `metastore` schema directly.

In [ ]:
# List all tables in the public schema of the metastore database
cursor.execute("""
    SELECT table_name
    FROM information_schema.tables
    WHERE table_schema = 'public'
    ORDER BY table_name;
""")
all_meta_tables = [row['table_name'] for row in cursor.fetchall()]
print(f'{len(all_meta_tables)} tables in the metastore database:')
for t in all_meta_tables:
    print(f'  - {t}')

## Close Connection

In [ ]:
cursor.close()
conn.close()
print('Connection closed.')